# LeetCode 4017 — Peaks in Array II bằng Python

- Link: https://leetcode.com/problems/peaks-in-array-ii/
- Độ khó: **Hard**
- Chủ đề: Cây phân đoạn (Segment Tree), Cây Fenwick (BIT), Cập nhật điểm + truy vấn đoạn, Đếm đoạn con theo cấu trúc "đỉnh"

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countOfPeaks(self, nums: list[int], queries: list[list[int]]) -> list[int]:
```

## 1. Giới thiệu & mục tiêu học

Cho mảng `nums` và các truy vấn hai loại:

- `[1, l, r]`: đếm số **đoạn con đỉnh** nằm trọn trong `nums[l..r]`.
- `[2, index, val]`: gán `nums[index] = val` (có hiệu lực cho các truy vấn sau).

Một đoạn con `nums[i..j]` là **đoạn con đỉnh** nếu độ dài `≥ 3` và có chỉ số `k` với `i < k < j` sao cho `nums[k] > nums[k-1]` và `nums[k] > nums[k+1]` (một **đỉnh** ở bên trong).

Với `n, q` tới `10^5`, ta không thể đếm lại mỗi lần. Chìa khoá gồm hai bước: (1) **đếm đoạn con theo đỉnh cuối cùng** để thu được công thức theo vị trí các đỉnh; (2) mỗi cập nhật chỉ đổi trạng thái "là đỉnh" của **tối đa 3 vị trí**, nên duy trì tập đỉnh bằng cây phân đoạn (hoặc Fenwick) với phép gộp hợp lý.

Sau notebook này bạn sẽ:

1. Định nghĩa "đỉnh" đúng (so sánh **chặt**, hai láng giềng, không tính hai đầu mảng).
2. Biến bài toán đếm đoạn con thành **tổng theo các đỉnh liên tiếp**: `Σ (p_t − l)·(p_{t+1} − p_t)`.
3. Dùng **cây phân đoạn** với nút lưu `(đỉnh đầu, đỉnh cuối, tổng S)` và phép gộp **không giao hoán** để trả lời mỗi truy vấn trong `O(log n)`.
4. Biết cách khác: Fenwick + danh sách đỉnh đã sắp (bisect).
5. Đối chiếu vét cạn nguyên văn / tuyến tính / Fenwick / segment tree bằng test ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.**

- `nums` độ dài `n`. Đoạn con `nums[i..j]` **là đoạn con đỉnh** nếu `j − i ≥ 2` và tồn tại `k` với `i < k < j`, `nums[k] > nums[k − 1]` và `nums[k] > nums[k + 1]`.
- Truy vấn `[1, l, r]`: số đoạn con đỉnh `nums[i..j]` với `l ≤ i` và `j ≤ r`.
- Truy vấn `[2, idx, val]`: gán `nums[idx] = val`.
- Trả về mảng kết quả cho các truy vấn loại 1 theo thứ tự xuất hiện.

**Ví dụ 1:** `nums = [1,3,2,4]`, `queries = [[1,0,3],[2,1,1],[1,0,3]]` → `[2, 0]`. Ban đầu chỉ `nums[1] = 3` là đỉnh: đoạn `[0..2]` và `[0..3]` chứa nó ở vị trí trong ⇒ 2. Sau khi gán `nums[1] = 1`, mảng `[1,1,2,4]` không còn đỉnh ⇒ 0.

**Ví dụ 2:** `nums = [9,8,9,8]`, `queries = [[1,1,3],[2,2,1],[1,0,2]]` → `[1, 0]`.

**Ví dụ 3:** `nums = [3,6,2,7,1]`, `queries = [[1,1,3],[2,3,0],[1,0,4]]` → `[0, 3]`.

**Ràng buộc**

- `3 <= n <= 10^5`, `0 <= nums[i] <= 10^5`
- `1 <= queries.length <= 10^5`
- `[1, l, r]` với `0 <= l < r <= n − 1`; `[2, index, val]` với `0 <= index <= n − 1`, `0 <= val <= 10^5`

In [1]:
import random, time, bisect

EXAMPLES = [
    ([1, 3, 2, 4], [[1, 0, 3], [2, 1, 1], [1, 0, 3]], [2, 0]),
    ([9, 8, 9, 8], [[1, 1, 3], [2, 2, 1], [1, 0, 2]], [1, 0]),
    ([3, 6, 2, 7, 1], [[1, 1, 3], [2, 3, 0], [1, 0, 4]], [0, 3]),
]
for nums, qs, e in EXAMPLES:
    print(f"nums={nums}, queries={qs} -> kỳ vọng {e}")

nums=[1, 3, 2, 4], queries=[[1, 0, 3], [2, 1, 1], [1, 0, 3]] -> kỳ vọng [2, 0]
nums=[9, 8, 9, 8], queries=[[1, 1, 3], [2, 2, 1], [1, 0, 2]] -> kỳ vọng [1, 0]
nums=[3, 6, 2, 7, 1], queries=[[1, 1, 3], [2, 3, 0], [1, 0, 4]] -> kỳ vọng [0, 3]


## 3. Kiến thức nền

### 3.1. "Đỉnh" tại vị trí `k`

`k` là **đỉnh** nếu `0 < k < n − 1` (có đủ hai láng giềng) và `nums[k] > nums[k−1]` **và** `nums[k] > nums[k+1]` (bất đẳng thức **chặt**: bằng nhau không tính). Hai đầu mảng không bao giờ là đỉnh.

In [2]:
def is_peak(a, k):
    return 0 < k < len(a) - 1 and a[k] > a[k - 1] and a[k] > a[k + 1]

a = [3, 6, 2, 7, 1, 1, 5, 5, 2]
print([k for k in range(len(a)) if is_peak(a, k)])      # 1, 3 (vị trí 6: 5 > 1 nhưng 5 == 5 -> không phải đỉnh)
print(is_peak([1, 1, 1], 1), is_peak([1, 2, 1], 1), is_peak([1, 2, 1], 0))

[1, 3]
False True False


### 3.2. Một cập nhật chỉ ảnh hưởng 3 vị trí

Trạng thái "là đỉnh" của `k` chỉ phụ thuộc `nums[k−1], nums[k], nums[k+1]`. Gán `nums[idx]` chỉ có thể đổi trạng thái của `idx − 1`, `idx`, `idx + 1`.

In [3]:
a = [1, 3, 2, 4, 3]
def peaks(a):
    return [k for k in range(len(a)) if is_peak(a, k)]
print(peaks(a))
a[2] = 9
print(peaks(a), "(chỉ các vị trí 1, 2, 3 có thể đổi)")

[1, 3]
[2] (chỉ các vị trí 1, 2, 3 có thể đổi)


### 3.3. Danh sách đã sắp và `bisect`

`bisect.bisect_left(P, x)` cho vị trí chèn `x` giữ `P` có thứ tự, `insort` chèn, `P.pop(i)` xoá. Tìm **đỉnh đầu tiên ≥ lo** và **đỉnh cuối cùng ≤ hi** bằng `bisect_left` và `bisect_right`.

In [4]:
P = [2, 5, 9, 14]
print(bisect.bisect_left(P, 5), bisect.bisect_right(P, 5), bisect.bisect_left(P, 6))
lo, hi = 3, 10
i = bisect.bisect_left(P, lo)               # chỉ số đỉnh đầu tiên >= lo
j = bisect.bisect_right(P, hi) - 1          # chỉ số đỉnh cuối cùng <= hi
print("các đỉnh trong [3, 10]:", P[i:j + 1])
bisect.insort(P, 7); print(P)
P.pop(bisect.bisect_left(P, 9)); print(P)

1 2 2
các đỉnh trong [3, 10]: [5, 9]
[2, 5, 7, 9, 14]
[2, 5, 7, 14]


### 3.4. Cây Fenwick (BIT) — cộng điểm, tổng tiền tố

BIT lưu `tree` cho phép `add(i, delta)` và `prefix(i)` trong `O(log n)`, nhờ `i & -i` (bit 1 thấp nhất).

In [5]:
class Fenwick:
    def __init__(self, n):
        self.n = n
        self.t = [0] * (n + 1)
    def add(self, i, delta):                  # i: 0-based
        i += 1
        while i <= self.n:
            self.t[i] += delta
            i += i & -i
    def prefix(self, i):                      # tổng a[0..i]; i = -1 -> 0
        i += 1
        s = 0
        while i > 0:
            s += self.t[i]
            i -= i & -i
        return s

bit = Fenwick(8)
for pos, v in [(1, 5), (3, 2), (6, 7)]:
    bit.add(pos, v)
print([bit.prefix(i) for i in range(8)], "| tổng [2..6] =", bit.prefix(6) - bit.prefix(1))

[0, 5, 5, 7, 7, 7, 14, 14] | tổng [2..6] = 9


### 3.5. Cây phân đoạn (segment tree) dạng mảng, gộp không giao hoán

Cây phân đoạn kích thước `size` (luỹ thừa 2) lưu `2·size` nút: lá `k` ở chỉ số `size + k`, cha của `i` là `i >> 1`, con là `2i` và `2i + 1`. Nút `i` tổng hợp đoạn `[tr, ph]` của hai con. Khi phép gộp **không giao hoán** (thứ tự trái → phải quan trọng, như bài này), truy vấn bottom-up phải gộp **các nút bên trái** vào accumulator trái theo thứ tự tăng và **các nút bên phải** vào accumulator phải theo thứ tự giảm, rồi ghép hai accumulator.

Ví dụ gọn: nút lưu `(đầu, cuối)` của dãy đánh dấu; gộp = (đầu của trái nếu có, cuối của phải nếu có).

In [6]:
size = 8
first = [-1] * (2 * size)
last = [-1] * (2 * size)
marks = {1, 4, 5}                        # các vị trí "được đánh dấu"
for k in marks:
    first[size + k] = last[size + k] = k
for i in range(size - 1, 0, -1):
    l, r = 2 * i, 2 * i + 1
    first[i] = first[l] if first[l] >= 0 else first[r]
    last[i] = last[r] if last[r] >= 0 else last[l]
print("gốc:", first[1], last[1])           # đầu = 1, cuối = 5
# nút cho đoạn [4, 7] là i = 3 (con phải của gốc)
print("đoạn [4,7]:", first[3], last[3])    # 4, 5

gốc: 1 5
đoạn [4,7]: 4 5


## 4. Ý tưởng nền tảng / trực giác

**Đếm đoạn con đỉnh bằng tay.** `nums = [1,5,2,6,3]`, hỏi `[l, r] = [0, 4]`. Đỉnh: vị trí `1` (5 > 1, 5 > 2) và `3` (6 > 2, 6 > 3); vị trí `2` thì không (2 < 5).

Các đoạn con độ dài `≥ 3`: `[0,2], [1,3], [2,4], [0,3], [1,4], [0,4]` (6 đoạn). Đoạn con cần có đỉnh ở vị trí **trong** (không phải hai đầu):

| Đoạn | Vị trí trong | Có đỉnh? |
|---|---|---|
| `[0,2]` | `{1}` | có (1) |
| `[1,3]` | `{2}` | không |
| `[2,4]` | `{3}` | có (3) |
| `[0,3]` | `{1,2}` | có (1) |
| `[1,4]` | `{2,3}` | có (3) |
| `[0,4]` | `{1,2,3}` | có |

Tổng **5**. Quan sát: đỉnh ở vị trí `1` hay `3` ở **đầu mút** của một đoạn con thì **không** tính cho đoạn đó (ví dụ đoạn `[1,3]` bắt đầu đúng tại đỉnh 1 nhưng đỉnh 1 không nằm trong).

## 5. Cách 1 — Vét cạn: đếm từng đoạn con

**Cách 1a — nguyên văn đề.** Với truy vấn `[l, r]`, thử mọi cặp `(i, j)` với `l ≤ i`, `i + 2 ≤ j ≤ r`, và mọi `k ∈ (i, j)` xem có là đỉnh không. `O(len³)` — chỉ để làm trọng tài nhỏ.

**Cách 1b — tuyến tính theo truy vấn.** Quét `j` từ trái sang phải, nhớ `q` = đỉnh gần nhất tính đến `j − 1`. Số `i` hợp lệ với `j` này là `q − l` nếu `q ≥ l + 1` (xem phần 6). Mỗi truy vấn `O(r − l)`.

In [7]:
def count_literal(a, l, r):
    total = 0
    for i in range(l, r + 1):
        for j in range(i + 2, r + 1):
            if any(is_peak(a, k) for k in range(i + 1, j)):
                total += 1
    return total

def count_linear(a, l, r):
    total, q = 0, -1
    for j in range(l + 2, r + 1):
        k = j - 1
        if k >= l + 1 and is_peak(a, k):
            q = k                                 # đỉnh gần nhất trong [l+1, j-1]
        if q >= l + 1:
            total += q - l                        # các i trong [l, q-1]
    return total

def run_queries(nums, queries, count):
    a = nums[:]
    out = []
    for q in queries:
        if q[0] == 1:
            out.append(count(a, q[1], q[2]))
        else:
            a[q[1]] = q[2]
    return out

for nums, qs, e in EXAMPLES:
    print(run_queries(nums, qs, count_literal), run_queries(nums, qs, count_linear), e)

[2, 0] [2, 0] [2, 0]
[1, 0] [1, 0] [1, 0]
[0, 3] [0, 3] [0, 3]


**Độ phức tạp.** 1a: `O(q · n³)`; 1b: `O(q · n)` (tới `10^10` với `n = q = 10^5`). **Nút thắt:** mỗi truy vấn quét lại cả đoạn dù chỉ vài vị trí thay đổi.

## 6. Bước đột phá

### 6.1. Đếm theo "đỉnh gần nhất bên trái của `j`"

Cố định `j` (đầu mút phải của đoạn con). Gọi `q(j)` là **đỉnh lớn nhất** `≤ j − 1` (đỉnh toàn cục). Đoạn `[i, j]` là đoạn con đỉnh ⇔ có đỉnh trong `[i + 1, j − 1]` ⇔ `q(j) ≥ i + 1` (vì `q(j)` là đỉnh gần `j` nhất, nếu có đỉnh nào trong khoảng thì `q(j)` cũng ≥ nó). Với giới hạn `i ≥ l`: số `i` hợp lệ là `max(0, q(j) − l)`.

### 6.2. Gom theo từng đỉnh

Gọi `p_1 < p_2 < … < p_m` là các đỉnh nằm trong **`[l + 1, r − 1]`**. Với `j` từ `l + 2` đến `r`:

- `q(j) = p_t` khi `j ∈ [p_t + 1, p_{t+1}]` (số lượng `p_{t+1} − p_t`), và với đỉnh cuối `j ∈ [p_m + 1, r]` (số lượng `r − p_m`).

Do đó

`đáp án = Σ_{t < m} (p_t − l)·(p_{t+1} − p_t) + (p_m − l)·(r − p_m)`  (và `0` nếu `m = 0`).

### 6.3. Tách `l` ra ngoài để gộp được

`Σ_{t<m} (p_t − l)(p_{t+1} − p_t) = Σ_{t<m} p_t(p_{t+1} − p_t) − l · (p_m − p_1)` (vì `Σ(p_{t+1} − p_t)` co dãn thành `p_m − p_1`).

Đặt `S = Σ_{t<m} p_t(p_{t+1} − p_t)` — một đại lượng chỉ phụ thuộc **các đỉnh liên tiếp**. Khi đó

`đáp án = S − l·(p_m − p_1) + (p_m − l)·(r − p_m)`, với `p_1`, `p_m` là đỉnh đầu và đỉnh cuối trong `[l + 1, r − 1]`.

### 6.4. Phép gộp cho cây phân đoạn

Mỗi nút lưu `(first, last, S)` cho các đỉnh trong đoạn của nút (`first = −1` nếu không có đỉnh).
`merge(A, B)`:

- nếu `A` rỗng ⇒ `B`; nếu `B` rỗng ⇒ `A`;
- ngược lại `(A.first, B.last, A.S + B.S + A.last·(B.first − A.last))` — số hạng cuối là cặp đỉnh liên tiếp **nối** hai nút.

**Bất biến.** `S` của một nút = `Σ p_t(p_{t+1} − p_t)` trên các cặp đỉnh liên tiếp **cả hai đều trong nút**.

**Quy tắc nếu… thì…:** cập nhật `nums[idx]` ⇒ chỉ tính lại trạng thái đỉnh ở `idx−1, idx, idx+1` rồi gộp lại các tổ tiên.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Fenwick + danh sách đỉnh đã sắp

**Giải thích từng đoạn.**

1. Giữ `P` = danh sách đỉnh sắp tăng (dùng `bisect`).
2. Với mỗi đỉnh `p` lưu `V[p] = p · (next(p) − p)` với `next(p)` là đỉnh kế tiếp **toàn cục** (nếu không có thì `V[p] = 0`). Fenwick giữ tổng các `V` theo vị trí.
3. Khi một vị trí **trở thành** đỉnh (hoặc **hết** là đỉnh): chèn/xoá khỏi `P`, cập nhật `V` của chính nó và của **đỉnh đứng ngay trước** (vì `next` của nó đổi).
4. Truy vấn `[l, r]`: `p_1` = đỉnh đầu ≥ `l+1`, `p_m` = đỉnh cuối ≤ `r−1` (bằng `bisect`). `S = Σ V` trên các đỉnh trong `[p_1, p_m − 1]` (đều có `next` trong khoảng). Rồi áp công thức 6.3.

In [8]:
class PeaksFenwick:
    def __init__(self, nums):
        self.a = nums[:]
        self.n = len(nums)
        self.P = []                               # danh sách đỉnh (sắp tăng)
        self.V = [0] * self.n
        self.bit = Fenwick(self.n)
        for k in range(1, self.n - 1):
            if is_peak(self.a, k):
                self._insert(k)

    def _set_v(self, p, newv):
        if self.V[p] != newv:
            self.bit.add(p, newv - self.V[p])
            self.V[p] = newv

    def _insert(self, k):
        i = bisect.bisect_left(self.P, k)
        pred = self.P[i - 1] if i > 0 else None
        succ = self.P[i] if i < len(self.P) else None
        self.P.insert(i, k)
        self._set_v(k, k * (succ - k) if succ is not None else 0)
        if pred is not None:
            self._set_v(pred, pred * (k - pred))

    def _remove(self, k):
        i = bisect.bisect_left(self.P, k)
        pred = self.P[i - 1] if i > 0 else None
        succ = self.P[i + 1] if i + 1 < len(self.P) else None
        self.P.pop(i)
        self._set_v(k, 0)
        if pred is not None:
            self._set_v(pred, pred * (succ - pred) if succ is not None else 0)

    def update(self, idx, val):
        ks = [k for k in (idx - 1, idx, idx + 1) if 1 <= k <= self.n - 2]
        before = {k: is_peak(self.a, k) for k in ks}
        self.a[idx] = val
        for k in ks:
            now = is_peak(self.a, k)
            if now and not before[k]:
                self._insert(k)
            elif before[k] and not now:
                self._remove(k)

    def query(self, l, r):
        lo, hi = l + 1, r - 1
        i = bisect.bisect_left(self.P, lo)
        j = bisect.bisect_right(self.P, hi) - 1
        if i > j:
            return 0
        p1, pm = self.P[i], self.P[j]
        S = self.bit.prefix(pm - 1) - self.bit.prefix(p1 - 1)
        return S - l * (pm - p1) + (pm - l) * (r - pm)

def peaks_fenwick(nums, queries):
    st = PeaksFenwick(nums)
    out = []
    for q in queries:
        if q[0] == 1:
            out.append(st.query(q[1], q[2]))
        else:
            st.update(q[1], q[2])
    return out

for nums, qs, e in EXAMPLES:
    print(peaks_fenwick(nums, qs), e)

[2, 0] [2, 0]
[1, 0] [1, 0]
[0, 3] [0, 3]


**Độ phức tạp.** Mỗi cập nhật: tối đa 3 lần chèn/xoá; `list.insert/pop` tốn `O(|P|)` nhưng là `memmove` rất nhanh; mỗi lần chèn/xoá thêm hai cập nhật Fenwick `O(log n)`. Mỗi truy vấn: hai `bisect` + hai `prefix`. Tổng khoảng `O((n + q) log n)` (cộng chi phí `memmove`).

### 7.2. Cách 3 — Cây phân đoạn `(first, last, S)` (bản nộp)

**Giải thích từng đoạn.**

1. Ba mảng `F, L, S` kích thước `2·size`. Lá `k` (`1 ≤ k ≤ n − 2`) nếu là đỉnh: `F = L = k`, `S = 0`; ngược lại `F = L = −1`.
2. `pull(i)` gộp hai con theo 6.4.
3. **Cập nhật** `nums[idx]`: tính lại các lá `idx−1..idx+1` (một đoạn liên tiếp), rồi lên từng tầng: tầng cha của đoạn lá `[lo, hi]` là `[lo>>1, hi>>1]` (tối đa 2 nút) — gọi `pull` cho các nút đó.
4. **Truy vấn** `[l, r]`: lấy kết quả gộp trên đoạn đỉnh `[l+1, r−1]` bằng cây bottom-up: accumulator trái gộp các nút bên trái theo thứ tự tăng, accumulator phải gộp nút bên phải theo thứ tự giảm, cuối cùng ghép hai accumulator. Sau đó áp công thức 6.3.

In [9]:
class PeakSegmentTree:
    def __init__(self, nums):
        self.a = nums[:]
        n = self.n = len(nums)
        size = 1
        while size < n:
            size <<= 1
        self.size = size
        self.F = [-1] * (2 * size)      # đỉnh đầu trong nút
        self.L = [-1] * (2 * size)      # đỉnh cuối trong nút
        self.S = [0] * (2 * size)       # tổng p_t * (p_{t+1} - p_t) trong nút
        for k in range(1, n - 1):
            if is_peak(self.a, k):
                self.F[size + k] = self.L[size + k] = k
        for i in range(size - 1, 0, -1):
            self._pull(i)

    def _pull(self, i):
        F, L, S = self.F, self.L, self.S
        l, r = 2 * i, 2 * i + 1
        if F[l] < 0:
            F[i], L[i], S[i] = F[r], L[r], S[r]
        elif F[r] < 0:
            F[i], L[i], S[i] = F[l], L[l], S[l]
        else:
            F[i] = F[l]
            L[i] = L[r]
            S[i] = S[l] + S[r] + L[l] * (F[r] - L[l])

    def update(self, idx, val):
        self.a[idx] = val
        lo, hi = max(1, idx - 1), min(self.n - 2, idx + 1)
        if lo > hi:
            return
        for k in range(lo, hi + 1):
            v = self.size + k
            self.F[v] = self.L[v] = k if is_peak(self.a, k) else -1
        a, b = (lo + self.size) >> 1, (hi + self.size) >> 1
        while a >= 1:
            for i in range(a, b + 1):
                self._pull(i)
            a >>= 1
            b >>= 1

    def query(self, l, r):
        lo, hi = l + 1, r - 1
        if lo > hi:
            return 0
        F, L, S = self.F, self.L, self.S
        lf = ll = -1; lS = 0                  # accumulator trái
        rf = rl = -1; rS = 0                  # accumulator phải
        a, b = lo + self.size, hi + self.size + 1
        while a < b:
            if a & 1:                         # gộp nút a vào BÊN PHẢI của accumulator trái
                if F[a] >= 0:
                    if lf < 0:
                        lf, ll, lS = F[a], L[a], S[a]
                    else:
                        lS = lS + S[a] + ll * (F[a] - ll)
                        ll = L[a]
                a += 1
            if b & 1:                         # gộp nút b-1 vào BÊN TRÁI của accumulator phải
                b -= 1
                if F[b] >= 0:
                    if rf < 0:
                        rf, rl, rS = F[b], L[b], S[b]
                    else:
                        rS = S[b] + rS + L[b] * (rf - L[b])
                        rf = F[b]
            a >>= 1
            b >>= 1
        if lf < 0:
            f, last, s = rf, rl, rS
        elif rf < 0:
            f, last, s = lf, ll, lS
        else:
            f, last, s = lf, rl, lS + rS + ll * (rf - ll)
        if f < 0:
            return 0
        return s - l * (last - f) + (last - l) * (r - last)

def peaks_segment_tree(nums, queries):
    st = PeakSegmentTree(nums)
    out = []
    for q in queries:
        if q[0] == 1:
            out.append(st.query(q[1], q[2]))
        else:
            st.update(q[1], q[2])
    return out

for nums, qs, e in EXAMPLES:
    print(peaks_segment_tree(nums, qs), e)

[2, 0] [2, 0]
[1, 0] [1, 0]
[0, 3] [0, 3]


**Độ phức tạp.** Dựng: `O(n)`. Mỗi cập nhật: tối đa 3 lá, mỗi tầng ≤ 2 nút ⇒ `O(log n)`. Mỗi truy vấn `O(log n)`. Tổng `O((n + q) log n)`, bộ nhớ `O(n)`.

## 8. TRACE từng bước

### 8.1. Công thức theo từng đỉnh cho `nums = [1,5,2,6,3,2,7,1]`, truy vấn `[0, 7]`

Đỉnh: vị trí `1` (5), `3` (6), `6` (7). Hàm trace liệt kê các đỉnh trong `[l+1, r−1]`, các số hạng `(p_t − l)(p_{t+1} − p_t)` và số hạng cuối `(p_m − l)(r − p_m)`; so với kết quả vét cạn.

In [10]:
def trace_query(a, l, r):
    ps = [k for k in range(l + 1, r) if is_peak(a, k)]
    print(f"nums[{l}..{r}] = {a[l:r + 1]}, đỉnh trong [{l + 1}, {r - 1}] = {ps}")
    if not ps:
        print("không có đỉnh -> 0")
        return 0
    total = 0
    for t in range(len(ps)):
        nxt = ps[t + 1] if t + 1 < len(ps) else r
        term = (ps[t] - l) * (nxt - ps[t])
        total += term
        print(f"  đỉnh p_{t + 1} = {ps[t]}: ({ps[t]} - {l}) * ({nxt} - {ps[t]}) = {term}   (tổng chạy = {total})")
    S = sum(ps[t] * (ps[t + 1] - ps[t]) for t in range(len(ps) - 1))
    formula = S - l * (ps[-1] - ps[0]) + (ps[-1] - l) * (r - ps[-1])
    print(f"  công thức gộp: S={S}, p_1={ps[0]}, p_m={ps[-1]} -> S - l*(p_m-p_1) + (p_m-l)*(r-p_m) = {formula}")
    print("  vét cạn nguyên văn:", count_literal(a, l, r))
    assert total == formula == count_literal(a, l, r)
    return total

trace_query([1, 5, 2, 6, 3, 2, 7, 1], 0, 7)
trace_query([1, 5, 2, 6, 3, 2, 7, 1], 2, 6)

nums[0..7] = [1, 5, 2, 6, 3, 2, 7, 1], đỉnh trong [1, 6] = [1, 3, 6]
  đỉnh p_1 = 1: (1 - 0) * (3 - 1) = 2   (tổng chạy = 2)
  đỉnh p_2 = 3: (3 - 0) * (6 - 3) = 9   (tổng chạy = 11)
  đỉnh p_3 = 6: (6 - 0) * (7 - 6) = 6   (tổng chạy = 17)
  công thức gộp: S=11, p_1=1, p_m=6 -> S - l*(p_m-p_1) + (p_m-l)*(r-p_m) = 17
  vét cạn nguyên văn: 17
nums[2..6] = [2, 6, 3, 2, 7], đỉnh trong [3, 5] = [3]
  đỉnh p_1 = 3: (3 - 2) * (6 - 3) = 3   (tổng chạy = 3)
  công thức gộp: S=0, p_1=3, p_m=3 -> S - l*(p_m-p_1) + (p_m-l)*(r-p_m) = 3
  vét cạn nguyên văn: 3


3

### 8.2. Trace cây phân đoạn: gộp hai nút

Với `nums = [1,5,2,6,3,2,7,1]`, `n = 8`: các lá đỉnh là `1, 3, 6`. Nút trái `(1, 3, S = 1·(3−1) = 2)` gộp nút phải `(6, 6, 0)` thành `(1, 6, 2 + 0 + 3·(6 − 3) = 11)`. Kiểm bằng công thức: `S = 1·(3−1) + 3·(6−3) = 2 + 9 = 11`. ✔

In [11]:
st = PeakSegmentTree([1, 5, 2, 6, 3, 2, 7, 1])
print("gốc: first, last, S =", st.F[1], st.L[1], st.S[1])
print("nút trái (vị trí 0..3):", st.F[2], st.L[2], st.S[2])
print("nút phải (vị trí 4..7):", st.F[3], st.L[3], st.S[3])
print("công thức đáp án [0,7]:", st.S[1] - 0 * (st.L[1] - st.F[1]) + (st.L[1] - 0) * (7 - st.L[1]), "| query:", st.query(0, 7))
st.update(2, 9)       # nums[2] = 9 làm vị trí 1 hết là đỉnh, vị trí 2 thành đỉnh
print("sau update(2, 9): gốc =", st.F[1], st.L[1], st.S[1], "| đỉnh thật:", [k for k in range(8) if is_peak(st.a, k)])

gốc: first, last, S = 1 6 11
nút trái (vị trí 0..3): 1 3 2
nút phải (vị trí 4..7): 6 6 0
công thức đáp án [0,7]: 17 | query: 17
sau update(2, 9): gốc = 2 6 8 | đỉnh thật: [2, 6]


## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (segment tree). Ta gói tất cả trong phương thức `countOfPeaks`; hàm `is_peak` viết thành hàm lồng nhau.

In [12]:
class Solution:
    def countOfPeaks(self, nums: list[int], queries: list[list[int]]) -> list[int]:
        n = len(nums)
        size = 1
        while size < n:
            size <<= 1
        F = [-1] * (2 * size)       # đỉnh đầu trong nút
        L = [-1] * (2 * size)       # đỉnh cuối trong nút
        S = [0] * (2 * size)        # tổng p_t * (p_{t+1} - p_t) trong nút

        def peak(k):
            return 0 < k < n - 1 and nums[k] > nums[k - 1] and nums[k] > nums[k + 1]

        def pull(i):
            l, r = 2 * i, 2 * i + 1
            if F[l] < 0:
                F[i], L[i], S[i] = F[r], L[r], S[r]
            elif F[r] < 0:
                F[i], L[i], S[i] = F[l], L[l], S[l]
            else:
                F[i] = F[l]
                L[i] = L[r]
                S[i] = S[l] + S[r] + L[l] * (F[r] - L[l])

        for k in range(1, n - 1):
            if peak(k):
                F[size + k] = L[size + k] = k
        for i in range(size - 1, 0, -1):
            pull(i)

        ans = []
        for q in queries:
            if q[0] == 1:
                l, r = q[1], q[2]
                lo, hi = l + 1, r - 1
                if lo > hi:
                    ans.append(0)
                    continue
                lf = ll = -1; lS = 0
                rf = rl = -1; rS = 0
                a, b = lo + size, hi + size + 1
                while a < b:
                    if a & 1:
                        if F[a] >= 0:
                            if lf < 0:
                                lf, ll, lS = F[a], L[a], S[a]
                            else:
                                lS += S[a] + ll * (F[a] - ll)
                                ll = L[a]
                        a += 1
                    if b & 1:
                        b -= 1
                        if F[b] >= 0:
                            if rf < 0:
                                rf, rl, rS = F[b], L[b], S[b]
                            else:
                                rS = S[b] + rS + L[b] * (rf - L[b])
                                rf = F[b]
                    a >>= 1
                    b >>= 1
                if lf < 0:
                    f, last, s = rf, rl, rS
                elif rf < 0:
                    f, last, s = lf, ll, lS
                else:
                    f, last, s = lf, rl, lS + rS + ll * (rf - ll)
                ans.append(0 if f < 0 else s - l * (last - f) + (last - l) * (r - last))
            else:
                idx, val = q[1], q[2]
                nums[idx] = val
                lo, hi = max(1, idx - 1), min(n - 2, idx + 1)
                if lo > hi:
                    continue
                for k in range(lo, hi + 1):
                    F[size + k] = L[size + k] = k if peak(k) else -1
                a, b = (lo + size) >> 1, (hi + size) >> 1
                while a >= 1:
                    for i in range(a, b + 1):
                        pull(i)
                    a >>= 1
                    b >>= 1
        return ans


for nums, qs, e in EXAMPLES:
    assert Solution().countOfPeaks(nums[:], [q[:] for q in qs]) == e, (nums, qs)
print("3 ví dụ của đề: OK")

3 ví dụ của đề: OK


## 10. Vì sao thuật toán đúng

**(a) Công thức đếm.** Phần 6.1–6.2: với mỗi `j`, các `i` hợp lệ là `l ≤ i ≤ q(j) − 1` với `q(j)` là đỉnh lớn nhất `≤ j − 1`; chỉ đỉnh `≥ l + 1` mới tính (nếu `q(j) ≤ l` thì không có `i` nào). Gom theo đỉnh `p_t` trong `[l + 1, r − 1]`: các `j` thuộc `[p_t + 1, p_{t+1}]` (hoặc `[p_m + 1, r]`) có `q(j) = p_t`, mỗi `j` cho `p_t − l` giá trị `i`. Cộng lại ra công thức. Kiểm bằng phép đếm bằng tay ở phần 4 (`5`) và bằng `count_literal` (nguyên văn đề) trong các test.

**(b) Khai triển `l`.** `Σ_{t<m}(p_t − l)(p_{t+1} − p_t) = S − l(p_m − p_1)` vì tổng các hiệu liên tiếp co dãn.

**(c) Phép gộp đúng.** `S` là tổng trên **các cặp đỉnh liên tiếp**. Gộp hai nút trái/phải, mọi cặp liên tiếp nằm trong một nút hoặc là cặp **nối** `(last của trái, first của phải)` — duy nhất một cặp, được cộng thêm `L_trái·(F_phải − L_trái)`. Nếu một bên rỗng thì cặp nối không tồn tại.

**(d) Truy vấn bottom-up đúng thứ tự.** Các nút chọn từ trái được thêm vào accumulator trái theo thứ tự **tăng dần vị trí**; các nút chọn từ phải được thêm vào **đầu** accumulator phải theo thứ tự giảm dần vị trí. Ghép cuối cùng accumulator trái với phải đúng thứ tự trái → phải.

**(e) Cập nhật đúng.** Trạng thái đỉnh của `k` chỉ phụ thuộc `k−1, k, k+1`, nên đổi `nums[idx]` chỉ đổi `idx−1, idx, idx+1`. Ta tính lại các lá đó và các tổ tiên của chúng.

**(f) Biên.** `k = 0` và `k = n − 1` không bao giờ là đỉnh; truy vấn có `r = l + 1` cho đoạn đỉnh rỗng ⇒ 0.

## 11. Phân tích Time/Space

Đặt `n` là độ dài mảng, `q` là số truy vấn.

| Cách | Cập nhật | Truy vấn | Space |
|---|---|---|---|
| 1a. Vét cạn | `O(1)` | `O(len³)` | `O(1)` |
| 1b. Tuyến tính | `O(1)` | `O(len)` | `O(1)` |
| 2. Fenwick + danh sách đỉnh | `O(log n)` (+ `memmove`) | `O(log n)` | `O(n)` |
| 3. Segment tree | `O(log n)` | `O(log n)` | `O(n)` |

Ta đo trên `n = q = 10^5`.

In [13]:
rng = random.Random(4017)
n = q = 100000
nums = [rng.randint(0, 100000) for _ in range(n)]
queries = []
for _ in range(q):
    if rng.random() < 0.5:
        l = rng.randint(0, n - 2); r = rng.randint(l + 1, n - 1)
        queries.append([1, l, r])
    else:
        queries.append([2, rng.randint(0, n - 1), rng.randint(0, 100000)])
t = time.time(); r_seg = Solution().countOfPeaks(nums[:], [x[:] for x in queries]); t1 = time.time() - t
t = time.time(); r_fen = peaks_fenwick(nums, queries); t2 = time.time() - t
print(f"segment tree: {t1:.2f}s | Fenwick: {t2:.2f}s | khớp: {r_seg == r_fen} | tổng truy vấn loại 1: {len(r_seg)}")
assert r_seg == r_fen

# Dạng xấu: mảng răng cưa (nhiều đỉnh), cập nhật liên tục
saw = [(i % 2) * 10 + (i % 7) for i in range(n)]
qs2 = [[2, rng.randint(0, n - 1), rng.randint(0, 20)] if i % 2 else [1, rng.randint(0, 1000), rng.randint(n - 1000, n - 1)] for i in range(q)]
t = time.time(); a1 = Solution().countOfPeaks(saw[:], [x[:] for x in qs2]); print(f"răng cưa: {time.time() - t:.2f}s")
assert a1 == peaks_fenwick(saw, qs2)

segment tree: 0.74s | Fenwick: 0.54s | khớp: True | tổng truy vấn loại 1: 49910


răng cưa: 0.72s


## 12. Bộ test

Ví dụ đề, edge case, rồi đối chiếu 4 cách (nguyên văn, tuyến tính, Fenwick, segment tree + `Solution`) trên input ngẫu nhiên nhỏ có seed. Giá trị nhỏ (`0..5`) để nhiều đỉnh và nhiều trường hợp **bằng nhau** (kiểm tra so sánh chặt).

In [14]:
def run_all(nums, qs):
    return {
        "literal": run_queries(nums, qs, count_literal),
        "linear": run_queries(nums, qs, count_linear),
        "fenwick": peaks_fenwick(nums, qs),
        "segtree": peaks_segment_tree(nums, qs),
        "Solution": Solution().countOfPeaks(nums[:], [x[:] for x in qs]),
    }

EDGE = [
    ([1, 2, 1], [[1, 0, 2]], [1]),                       # đúng một đoạn con, đỉnh ở giữa
    ([1, 1, 1], [[1, 0, 2]], [0]),                       # bằng nhau không phải đỉnh
    ([1, 2, 3], [[1, 0, 2]], [0]),                       # tăng dần
    ([3, 2, 1], [[1, 0, 2]], [0]),
    ([1, 2, 1, 2, 1], [[1, 0, 4]], [5]),                 # hai đỉnh (vị trí 1 và 3): 2 + 3 = 5
    ([1, 3, 1], [[1, 0, 1], [1, 1, 2]], [0, 0]),         # đoạn quá ngắn (độ dài 2)
    ([5, 1, 5], [[1, 0, 2], [2, 1, 9], [1, 0, 2], [2, 0, 0], [1, 0, 2]], [0, 1, 1]),   # sau khi gán 9 ở giữa mới có đỉnh
]
for nums, qs, e in EXAMPLES + EDGE:
    ref = run_queries(nums, qs, count_literal)
    assert ref == e, ("trọng tài lệch kỳ vọng", nums, qs, ref, e)
    res = run_all(nums, qs)
    for name, out in res.items():
        assert out == e, (name, nums, qs, out, e)
print("Ví dụ đề + edge case: mọi cách đúng")

rng = random.Random(401701)
for _ in range(800):
    n = rng.randint(3, 12)
    nums = [rng.randint(0, 5) for _ in range(n)]
    qs = []
    for _ in range(rng.randint(1, 12)):
        if rng.random() < 0.5:
            l = rng.randint(0, n - 2); r = rng.randint(l + 1, n - 1)
            qs.append([1, l, r])
        else:
            qs.append([2, rng.randint(0, n - 1), rng.randint(0, 5)])
    res = run_all(nums, qs)
    ref = res["literal"]
    for name, out in res.items():
        assert out == ref, (name, nums, qs, out, ref)
print("800 input nhỏ ngẫu nhiên: nguyên văn = tuyến tính = Fenwick = segment tree = Solution")

Ví dụ đề + edge case: mọi cách đúng
800 input nhỏ ngẫu nhiên: nguyên văn = tuyến tính = Fenwick = segment tree = Solution


In [15]:
rng = random.Random(7)
for _ in range(60):
    n = rng.randint(30, 300)
    nums = [rng.randint(0, 20) for _ in range(n)]
    qs = []
    for _ in range(100):
        if rng.random() < 0.5:
            l = rng.randint(0, n - 2); r = rng.randint(l + 1, n - 1)
            qs.append([1, l, r])
        else:
            qs.append([2, rng.randint(0, n - 1), rng.randint(0, 20)])
    ref = run_queries(nums, qs, count_linear)
    assert peaks_fenwick(nums, qs) == ref == peaks_segment_tree(nums, qs) == Solution().countOfPeaks(nums[:], [x[:] for x in qs])
print("60 mảng 30..300: tuyến tính = Fenwick = segment tree = Solution")

# Input queries/nums gốc không bị hỏng khi dùng bản sao; Solution tự sửa nums nhưng không sửa queries
qs = [[1, 0, 3], [2, 1, 1], [1, 0, 3]]
snap = [x[:] for x in qs]
Solution().countOfPeaks([1, 3, 2, 4], qs)
assert qs == snap

60 mảng 30..300: tuyến tính = Fenwick = segment tree = Solution


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Coi đỉnh ở **đầu mút** của đoạn con là hợp lệ

**Code sai:** đếm mọi đoạn `[i, j]` có đỉnh ở `[i, j]` (cả hai đầu).
**Vì sao sai:** đề bắt buộc `i < k < j`. Với `nums = [1,3,2]` và đoạn `[1..2]` hay `[0..1]` không hợp lệ (độ dài 2).
**Code đúng:** chỉ xét đỉnh trong `[l + 1, r − 1]`.

In [16]:
def wrong_count_peaks_in_range(a, l, r):
    # SAI: lấy số đỉnh trong [l, r] rồi nhân... ví dụ: đếm đoạn con chứa đỉnh ở cả hai đầu
    total = 0
    for i in range(l, r + 1):
        for j in range(i + 2, r + 1):
            if any(is_peak(a, k) for k in range(i, j + 1)):        # SAI: i..j thay vì i+1..j-1
                total += 1
    return total

a = [1, 3, 2, 4]
print("sai:", wrong_count_peaks_in_range(a, 0, 3), "| đúng:", count_literal(a, 0, 3))
a = [2, 1, 3, 1]
print("sai:", wrong_count_peaks_in_range(a, 0, 3), "| đúng:", count_literal(a, 0, 3))      # đoạn [0,2] kết thúc ngay tại đỉnh 2 nên không hợp lệ

sai: 3 | đúng: 2
sai: 3 | đúng: 2


### Lỗi 2 — So sánh không chặt (`>=`)

**Code sai:** `nums[k] >= nums[k-1] and nums[k] >= nums[k+1]`.
**Vì sao sai:** `[2,2,2]` không có đỉnh. **Code đúng:** `>` cho cả hai.

### Lỗi 3 — Quên rằng cập nhật ảnh hưởng **cả hai láng giềng**

**Code sai:** chỉ tính lại trạng thái đỉnh tại `idx`.
**Vì sao sai:** `idx−1` và `idx+1` cũng phụ thuộc `nums[idx]`.
**Code đúng:** tính lại `idx−1, idx, idx+1` (nhớ kiểm tra biên `1..n−2`).

In [17]:
def peaks_segtree_wrong_update(nums, queries):
    st = PeakSegmentTree(nums)
    out = []
    for q in queries:
        if q[0] == 1:
            out.append(st.query(q[1], q[2]))
        else:
            idx = q[1]
            st.a[idx] = q[2]
            # SAI: chỉ cập nhật lá idx, quên idx-1 và idx+1
            if 1 <= idx <= st.n - 2:
                v = st.size + idx
                st.F[v] = st.L[v] = idx if is_peak(st.a, idx) else -1
                v >>= 1
                while v >= 1:
                    st._pull(v); v >>= 1
    return out

# Phản ví dụ: gán nums[4] = 4 làm vị trí 3 (giá trị 4) HẾT là đỉnh, nhưng cây chỉ cập nhật lá 4
nums = [2, 3, 2, 4, 0]
qs = [[2, 4, 4], [1, 1, 4]]
print("sai:", peaks_segtree_wrong_update(nums, qs), "| đúng:", run_queries(nums, qs, count_literal))

sai: [2] | đúng: [0]


### Lỗi 4 — Gộp truy vấn bottom-up sai thứ tự (phép gộp không giao hoán)

**Code sai:** gom mọi nút vào **một** accumulator theo thứ tự gặp (trái, phải xen kẽ).
**Vì sao sai:** `merge(A, B) ≠ merge(B, A)` vì số hạng nối `A.last·(B.first − A.last)` phụ thuộc thứ tự. **Code đúng:** hai accumulator, nút trái thêm vào cuối accumulator trái, nút phải thêm vào **đầu** accumulator phải.

### Lỗi 5 — Dùng `S` mà quên trừ `l·(p_m − p_1)` hoặc quên số hạng cuối `(p_m − l)(r − p_m)`

**Vì sao sai:** `S` chỉ chứa `p_t(p_{t+1} − p_t)`; thiếu `−l` gây sai khi `l > 0`; thiếu số hạng cuối bỏ qua các `j` sau đỉnh cuối.

In [18]:
nums = [0, 1, 5, 2, 6, 3, 0]
l, r = 1, 6                         # l > 0 và có hai đỉnh (vị trí 2 và 4) để lộ lỗi
ps = [k for k in range(l + 1, r) if is_peak(nums, k)]
S = sum(ps[t] * (ps[t + 1] - ps[t]) for t in range(len(ps) - 1))
print("thiếu -l*(p_m-p_1):", S + (ps[-1] - l) * (r - ps[-1]), "| thiếu số hạng cuối:", S - l * (ps[-1] - ps[0]),
      "| đúng:", S - l * (ps[-1] - ps[0]) + (ps[-1] - l) * (r - ps[-1]), "| vét cạn:", count_literal(nums, l, r))

thiếu -l*(p_m-p_1): 10 | thiếu số hạng cuối: 2 | đúng: 8 | vét cạn: 8


## 14. Mẹo & nhận dạng pattern

- **"Đếm đoạn con có tính chất xuất hiện ít nhất một lần"** → đếm theo **phần tử xuất hiện cuối cùng/đầu tiên** thay vì đếm bù.
- **Cập nhật điểm ảnh hưởng vài vị trí lân cận** → cây phân đoạn / Fenwick + cập nhật cục bộ.
- **Phép gộp không giao hoán** → lưu tóm tắt `(đầu, cuối, tổng)` và nhớ **số hạng nối** ở ranh giới.
- Tách tham số truy vấn (`l`) khỏi tổng tiền xử lý để nút chỉ lưu đại lượng không phụ thuộc truy vấn.
- Với `n = 10^5` trong Python, dùng cây phân đoạn **bottom-up dạng mảng** thay đệ quy; cập nhật chỉ tối đa 2 nút mỗi tầng khi đổi một đoạn lá liên tiếp.
- Kiểm thử bằng vét cạn **nguyên văn đề** trên mảng giá trị nhỏ để có nhiều đỉnh và nhiều trường hợp bằng nhau.

## 15. Biến thể

1. **Peaks in Array I** (đếm **số đỉnh** trong `[l+1, r−1]` có cập nhật): Fenwick đếm (Bài 5).
2. **Đếm đoạn con có đúng `k` đỉnh:** cần trạng thái phong phú hơn (ví dụ cửa sổ trượt trên danh sách đỉnh).
3. **Thung lũng (valley)** thay cho đỉnh: đổi dấu `nums`.
4. **Đếm đoạn con có ít nhất một đỉnh VÀ một thung lũng:** gộp hai tập; dùng bù trừ.
5. **Truy vấn offline không cập nhật:** prefix sum trên các đỉnh, mỗi truy vấn `O(log)` bằng `bisect`.

## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Cập nhật / Truy vấn | Space | Khi nào dùng |
|---|---|---|---|---|
| 1a. Vét cạn | Mọi `(i, j, k)` | `O(1)` / `O(len³)` | `O(1)` | Trọng tài |
| 1b. Tuyến tính | Quét `j`, nhớ đỉnh gần nhất | `O(1)` / `O(len)` | `O(1)` | Trọng tài cho mảng vừa |
| 2. Fenwick + danh sách đỉnh | `V[p] = p·(next − p)` | `O(log n)` / `O(log n)` | `O(n)` | Khi thích `bisect` |
| 3. Segment tree `(first, last, S)` | Gộp không giao hoán | `O(log n)` / `O(log n)` | `O(n)` | **Bản nộp** |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `[k for k in range(5) if is_peak([2,5,3,6,1], k)]` bằng gì?

**Bài 2 (tính tay).** `nums = [1,5,2,6,3]`, truy vấn `[1, 0, 4]`. Dùng công thức `Σ(p_t − l)(p_{t+1} − p_t) + (p_m − l)(r − p_m)` (đỉnh `1` và `3`). Kết quả bao nhiêu?

**Bài 3 (điền TODO).** Hoàn thiện `merge(A, B)` cho nút `(first, last, S)` (dùng `−1` cho "rỗng").

**Bài 4 (tự viết lại).** Viết `my_count(a, l, r)` bằng công thức theo danh sách đỉnh (không dùng segment tree), tức `O(r − l)` mỗi truy vấn.

**Bài 5 (biến thể).** **Peaks in Array I:** hỗ trợ `update(idx, val)` và `count_peaks(l, r)` = số đỉnh trong `[l+1, r−1]`. Dùng Fenwick đếm.

In [19]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def merge(A, B):
    # A, B là (first, last, S); first == -1 nghĩa là rỗng
    # TODO: nếu A rỗng trả B; nếu B rỗng trả A; ngược lại (A.first, B.last, A.S + B.S + A.last*(B.first - A.last))
    pass

# Bài 4 — TODO
def my_count(a, l, r):
    # TODO: ps = các đỉnh trong [l+1, r-1]; áp dụng công thức; trả 0 nếu rỗng
    pass

# Bài 5 — TODO
class PeakCounter:
    def __init__(self, nums):
        # TODO: lưu nums, tạo Fenwick, thêm 1 tại mỗi đỉnh
        pass
    def update(self, idx, val):
        # TODO: tính lại trạng thái của idx-1, idx, idx+1 và cập nhật Fenwick theo chênh lệch
        pass
    def count_peaks(self, l, r):
        # TODO: tổng Fenwick trên [l+1, r-1]
        pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: kiểm tra từng `k` trong `1..3`.
- Mức 2: `5 > 2, 3` ✔; `3 > 5`? không; `6 > 3, 1` ✔.
- Mức 3: `[1, 3]`.

**Bài 2.**
- Mức 1: `l = 0`, `r = 4`, đỉnh `p_1 = 1`, `p_2 = 3`.
- Mức 2: số hạng thứ nhất `(1 − 0)(3 − 1) = 2`; số hạng cuối `(3 − 0)(4 − 3) = 3`.
- Mức 3: `2 + 3 = 5` (khớp phần 4).

**Bài 3.**
- Mức 1: hai trường hợp rỗng.
- Mức 2: số hạng nối `A.last * (B.first - A.last)`.
- Mức 3: `return (A[0], B[1], A[2] + B[2] + A[1] * (B[0] - A[1]))`.

**Bài 4.**
- Mức 1: lấy `ps` bằng list comprehension.
- Mức 2: `S = Σ ps[t] * (ps[t+1] - ps[t])`.
- Mức 3: `S - l * (ps[-1] - ps[0]) + (ps[-1] - l) * (r - ps[-1])`.

**Bài 5.**
- Mức 1: lưu trạng thái đỉnh hiện tại trong mảng `is_p`.
- Mức 2: khi một vị trí đổi trạng thái, `bit.add(k, ±1)`.
- Mức 3: `bit.prefix(r - 1) - bit.prefix(l)`.

## 19. Đáp án tham khảo

In [20]:
# Bài 1, 2
assert [k for k in range(5) if is_peak([2, 5, 3, 6, 1], k)] == [1, 3]
assert Solution().countOfPeaks([1, 5, 2, 6, 3], [[1, 0, 4]]) == [5]

# Bài 3
def merge_ans(A, B):
    if A[0] < 0:
        return B
    if B[0] < 0:
        return A
    return (A[0], B[1], A[2] + B[2] + A[1] * (B[0] - A[1]))

assert merge_ans((-1, -1, 0), (3, 3, 0)) == (3, 3, 0)
assert merge_ans((1, 3, 2), (6, 6, 0)) == (1, 6, 11)
assert merge_ans((1, 3, 2), (-1, -1, 0)) == (1, 3, 2)
print("Bài 1–3 OK")

Bài 1–3 OK


In [21]:
# Bài 4
def my_count_ans(a, l, r):
    ps = [k for k in range(l + 1, r) if is_peak(a, k)]
    if not ps:
        return 0
    S = sum(ps[t] * (ps[t + 1] - ps[t]) for t in range(len(ps) - 1))
    return S - l * (ps[-1] - ps[0]) + (ps[-1] - l) * (r - ps[-1])

# Bài 5
class PeakCounterAns:
    def __init__(self, nums):
        self.a = nums[:]
        self.n = len(nums)
        self.bit = Fenwick(self.n)
        self.is_p = [False] * self.n
        for k in range(1, self.n - 1):
            if is_peak(self.a, k):
                self.is_p[k] = True
                self.bit.add(k, 1)
    def update(self, idx, val):
        self.a[idx] = val
        for k in (idx - 1, idx, idx + 1):
            if 1 <= k <= self.n - 2:
                now = is_peak(self.a, k)
                if now != self.is_p[k]:
                    self.bit.add(k, 1 if now else -1)
                    self.is_p[k] = now
    def count_peaks(self, l, r):
        if l + 1 > r - 1:
            return 0
        return self.bit.prefix(r - 1) - self.bit.prefix(l)

rng = random.Random(99)
for _ in range(200):
    n = rng.randint(3, 15)
    a = [rng.randint(0, 6) for _ in range(n)]
    pc = PeakCounterAns(a)
    for _ in range(15):
        if rng.random() < 0.5:
            idx, val = rng.randint(0, n - 1), rng.randint(0, 6)
            a[idx] = val
            pc.update(idx, val)
        else:
            l = rng.randint(0, n - 2); r = rng.randint(l + 1, n - 1)
            assert my_count_ans(a, l, r) == count_literal(a, l, r)
            assert pc.count_peaks(l, r) == sum(1 for k in range(l + 1, r) if is_peak(a, k))
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template:**

```python
# đáp án truy vấn [l, r], các đỉnh p_1 < ... < p_m nằm trong [l+1, r-1]
S = sum(p_t * (p_{t+1} - p_t))                          # cặp đỉnh liên tiếp
ans = S - l * (p_m - p_1) + (p_m - l) * (r - p_m)       # 0 nếu không có đỉnh

# nút segment tree: (first, last, S); rỗng: first = -1
merge(A, B) = (A.first, B.last, A.S + B.S + A.last * (B.first - A.last))   # xử lý rỗng riêng
update(idx): tính lại idx-1, idx, idx+1 rồi pull các tổ tiên
query: accumulator trái (thêm vào cuối) + accumulator phải (thêm vào đầu), rồi ghép
```

**Checklist trước khi submit:**

- [ ] Đỉnh dùng so sánh **chặt** và chỉ ở `1..n-2`.
- [ ] Chỉ tính đỉnh trong `[l+1, r-1]`; `r = l + 1` cho 0.
- [ ] Cập nhật tính lại **ba** vị trí `idx−1, idx, idx+1` (cắt theo biên).
- [ ] Phép gộp xử lý nút rỗng và số hạng nối.
- [ ] Truy vấn bottom-up đúng thứ tự (hai accumulator).
- [ ] Công thức: `S − l·(p_m − p_1) + (p_m − l)·(r − p_m)`.
- [ ] Đối chiếu vét cạn nguyên văn đề trên mảng nhỏ giá trị bằng nhau.